# 04 — GASTAT preprocessing

Prepares the three GASTAT (General Authority for Statistics) files for the survival model:

| Part | File | What it gives the model |
|---|---|---|
| A | `gastat_gdp_….csv` | How fast the business's sector was growing **before** it started |
| B | `gastat_sme_operating_expenses_revenues_….csv` | How profitable small businesses in the sector were the year **before** it started |
| C | `National Classification … (SSIC) ….xlsx` | Whether the activity needs a license from an authority other than the Ministry of Commerce |
| D | (join test) | Attaches A–C to the businesses of `01_preprocessing` and checks the coverage |

Monsha'at is not used (its 2026 counts include our own surviving businesses: leakage). SAMA is prepared in its own notebook.

**Rules for this notebook**
- The raw files are never changed. Results are saved as new files in the data folder, never in Git.
- Nothing is guessed. A value that cannot be computed stays empty and gets a flag.
- **Leakage rule:** a business that starts in quarter Q may only get numbers from periods that end **before** Q.
  Every feature table has a `…_used` column (the period the number comes from) and an `assert` that proves it.
- Clear all outputs before committing.

**Each check follows the same pattern:** what we see → decision → result. Write the decision in the markdown cell
under the check after running it.

### Cell 1 — Setup
Finds the data folder. On Colab the files are downloaded into `/content/darb-data` (temporary copies, gone when the
session ends). On a laptop it reads a local copy of the Drive folder.

In [14]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd

IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in globals() else False

if IN_COLAB:
    %pip install -q gdown python-calamine
    DATA_DIR = Path('/content/darb-data')
else:
    # local copy of the Drive folder "Darb DataSets" (outside Git and OneDrive)
    DATA_DIR = Path(os.environ.get('DARB_DATA', str(Path.home() / 'darb-data')))

GASTAT_DIR = DATA_DIR / 'Gastat'
pd.set_option('display.width', 250, 'display.max_columns', 40, 'display.max_colwidth', 60)
print('Running on Colab' if IN_COLAB else 'Running locally', '| data folder:', DATA_DIR)

Running on Colab | data folder: /content/darb-data


### Cell 2 — The three GASTAT files
Same file names and Google Drive IDs as `02_data_dictionary`, so every notebook reads the same files.

In [15]:
GDP_FILE  = 'gastat_gdp_2026-10-21T09_14_09.657Z.csv'
SME_FILE  = 'gastat_sme_operating_expenses_revenues_2026-10-21T09_39_24.273Z.csv'
SSIC_FILE = 'National Classification for the Economic Activities(SSIC)_15 Sep_2026_AR.xlsx'
DRIVE_IDS = {GDP_FILE:  '1fRoZuqgt1ltaQcXfSzbI2rk9mjK1pRH8',
             SME_FILE:  '11WopaP1bfkU4bmeugGygvgxjorm56iE-',
             SSIC_FILE: '1wlS-84Ojsiv0UP8sjmZmNcENU_M8J0wZ'}

if IN_COLAB:
    import gdown
    GASTAT_DIR.mkdir(parents=True, exist_ok=True)
    for name, drive_id in DRIVE_IDS.items():
        if not (GASTAT_DIR / name).exists():
            print('downloading', name)
            gdown.download(id=drive_id, output=str(GASTAT_DIR / name), quiet=True)

for name in DRIVE_IDS:
    path = GASTAT_DIR / name
    print('found  ' if path.exists() else 'MISSING', name, f'({path.stat().st_size / 1e3:.0f} KB)' if path.exists() else '')

found   gastat_gdp_2026-10-21T09_14_09.657Z.csv (43 KB)
found   gastat_sme_operating_expenses_revenues_2026-10-21T09_39_24.273Z.csv (58 KB)
found   National Classification for the Economic Activities(SSIC)_15 Sep_2026_AR.xlsx (613 KB)


### Cell 3 — Helpers used by every part
- **`read_csv_raw` / `find_header` / `with_header`:** statistics files often start with a title or notes, so the
  file is read without a header and the real header row is found (the first row that is almost as full as the
  fullest one). Same method as `02_data_dictionary` Cell 3.
- **`to_number`:** turns text into numbers (removes thousands separators) and returns which cells failed, so
  nothing is turned into a number silently.
- **Quarters as numbers:** `2025Q2` ↔ `2025 × 4 + 1`, so "4 quarters earlier" is simply `− 4`.

In [16]:
SECTION_LETTERS = list('ABCDEFGHIJKLMNOPQRSTU')     # the 21 SSIC sections

def read_csv_raw(path):
    """The whole CSV as text, without a header, so title rows above the table can be seen."""
    for encoding in ['utf-8-sig', 'cp1256']:
        try:
            return pd.read_csv(path, header=None, dtype=str, encoding=encoding)
        except UnicodeDecodeError:
            continue
    raise ValueError(f'Could not read {path.name}')

def find_header(raw):
    """First row near the top that is at least 80% as full as the fullest row there."""
    top = raw.head(25).notna().sum(axis=1)
    return int(top.index[top >= 0.8 * top.max()][0]) if len(top) and top.max() > 0 else 0

def with_header(raw, header_row):
    """Use one row as column names; the rows below it are the data. Empty rows and empty unnamed columns go."""
    names = [re.sub(r'\s+', ' ', str(v)).strip() if pd.notna(v) else f'(no name {i})'
             for i, v in enumerate(raw.iloc[header_row], 1)]
    table = raw.iloc[header_row + 1:].copy()
    table.columns = names
    table = table.dropna(how='all')
    empty = [c for c in table.columns if c.startswith('(no name') and table[c].isna().all()]
    return table.drop(columns=empty).reset_index(drop=True)

def clean_text(values):
    """Strip spaces and the '.0' that Excel/CSV exports add to whole numbers ('3.0' -> '3')."""
    return values.fillna('').astype(str).str.strip().str.replace(r'\.0$', '', regex=True)

def to_number(values):
    """Text -> number. Returns (numbers, failed): failed = a filled cell that is not a number."""
    text = values.fillna('').astype(str).str.strip().str.replace(r'[,\s٬]', '', regex=True)
    empty = text.isin(['', '-', '..', '…', 'nan'])
    numbers = pd.to_numeric(text.where(~empty), errors='coerce').astype(float)
    return numbers, (~empty & numbers.isna())

def qlabel(n):
    """Quarter number -> label: 8101 -> '2025Q2'."""
    return f'{n // 4}Q{n % 4 + 1}'

def qnum(label):
    """Quarter label -> number: '2025Q2' -> 8101."""
    return int(label[:4]) * 4 + int(label[-1]) - 1

COHORTS = ['2024Q4', '2025Q1', '2025Q2', '2025Q3']   # the cohorts 01_preprocessing labels
print('Helpers ready')

Helpers ready


## Part A — GDP by economic activity

### Cell A1 — Read the file and check its columns
Expected columns (from `02_data_dictionary`): `Quarter ID`, `Quarter`, `Economic Activity Section ID`,
`Economic Activity Section`, `GDP`. Any **other** column is printed with its values: if there is, for example, a
price basis (current / constant prices), it must be chosen in Cell A2, or the same quarter is counted twice.

If the header guess is wrong, write the right row number in `GDP_HEADER` and run the cell again.

In [17]:
GDP_HEADER = None          # write a row number here only if the guess below is wrong

raw = read_csv_raw(GASTAT_DIR / GDP_FILE)
header = GDP_HEADER if GDP_HEADER is not None else find_header(raw)
print(f'Header row: {header}. Rows above it (titles, notes):')
print(raw.head(header).fillna('').to_string() if header else '   none')

gdp_raw = with_header(raw, header)
print(f'\n{len(gdp_raw):,} data rows | columns: {list(gdp_raw.columns)}')

GDP_COLUMNS = {'Quarter ID': 'quarter_id', 'Quarter': 'quarter_label',
               'Economic Activity Section ID': 'activity_id', 'Economic Activity Section': 'activity', 'GDP': 'gdp'}
missing = set(GDP_COLUMNS) - set(gdp_raw.columns)
assert not missing, f'Columns not found: {missing}. Check the header row (GDP_HEADER).'

extra = [c for c in gdp_raw.columns if c not in GDP_COLUMNS]
print('Other columns:', extra or 'none')
for column in extra:
    print(f'\n--- {column} ---')
    print(gdp_raw[column].value_counts(dropna=False).head(10).to_string())
gdp_raw.head(8)

Header row: 0. Rows above it (titles, notes):
   none

792 data rows | columns: ['Quarter ID', 'Quarter', 'Economic Activity Section ID', 'Economic Activity Section', 'GDP']
Other columns: none


,Quarter ID,Quarter,Economic Activity Section ID,Economic Activity Section,GDP
0,20101,2010-Q1,1,"Agriculture, Forestry & Fishing",18018.0
1,20101,2010-Q1,2,Mining & Quarrying,200976.0
2,20101,2010-Q1,3,Manufacturing,93406.0
3,20101,2010-Q1,4,"Electricity, Gas and Water",4027.0
4,20101,2010-Q1,5,Construction,37467.0
5,20101,2010-Q1,6,"Wholesale & Retail Trade, Restaurants & hotels",55691.0
6,20101,2010-Q1,7,"Transport, Storage & Communication",31173.0
7,20101,2010-Q1,8,"Finance, Insurance, Real Estate & Business Services",72160.0


**What we see →** _(fill in: header row, other columns, anything odd)_

**Decision →** _(fill in)_

### Cell A2 — Data types and quarters
1. `GDP` text → number. Cells that are not a number are counted and shown (never set to 0).
2. `Quarter ID` (e.g. `20101` = 2010 Q1) → a real quarter. It is checked against the `Quarter` label.
3. **Exact duplicate rows** (every column the same) get `flag_exact_duplicate` and are left out of the features.

If Cell A1 showed a price-basis column, keep one basis here (constant prices measure real growth; current prices
include inflation). Write the choice in `KEEP_ROWS`.

In [18]:
# Example if a price column exists: KEEP_ROWS = {'Price Basis': 'Constant Prices'}
KEEP_ROWS = {}

gdp = gdp_raw.copy()
for column, value in KEEP_ROWS.items():
    print(f'Keeping {column} = {value}: {int((gdp[column] == value).sum()):,} of {len(gdp):,} rows')
    gdp = gdp[gdp[column] == value]
gdp = gdp[list(GDP_COLUMNS)].rename(columns=GDP_COLUMNS).reset_index(drop=True)

for column in ['quarter_id', 'quarter_label', 'activity_id', 'activity']:
    gdp[column] = clean_text(gdp[column])
gdp['activity'] = gdp['activity'].str.replace(r'\s+', ' ', regex=True)

# 1. GDP -> number
gdp_text = gdp['gdp'].copy()
gdp['gdp'], failed = to_number(gdp['gdp'])
print(f'GDP cells that are not a number: {int(failed.sum())}')
if failed.any():
    print(gdp_text[failed].value_counts().head(10).to_string())
gdp['flag_gdp_missing'] = gdp['gdp'].isna()
print(f'GDP cells empty: {int(gdp["flag_gdp_missing"].sum())}')
print(f'GDP values zero or below: {int((gdp["gdp"] <= 0).sum())}')

# 2. Quarter ID -> quarter
parts = gdp['quarter_id'].str.extract(r'^(\d{4})([1-4])$')
gdp['flag_bad_quarter'] = parts.isna().any(axis=1)
gdp['qnum'] = (pd.to_numeric(parts[0]) * 4 + pd.to_numeric(parts[1]) - 1).astype('Int64')
gdp['quarter'] = gdp['qnum'].map(lambda n: qlabel(int(n)) if pd.notna(n) else '')
print(f'\nQuarter IDs not in the form YYYYQ: {int(gdp["flag_bad_quarter"].sum())}')
if gdp['flag_bad_quarter'].any():
    print(gdp.loc[gdp['flag_bad_quarter'], 'quarter_id'].value_counts().head(10).to_string())

digits = gdp['quarter_label'].str.replace(r'\D', '', regex=True)
year, q = parts[0].fillna(''), parts[1].fillna('')
label_agrees = (digits == year + q) | (digits == q + year)
print(f'Quarter label agrees with the ID: {label_agrees.mean():.1%}')
if not label_agrees.all():
    print('Examples that disagree:')
    print(gdp.loc[~label_agrees, ['quarter_id', 'quarter_label']].drop_duplicates().head(5).to_string(index=False))

# 3. Exact duplicate rows
gdp['flag_exact_duplicate'] = gdp.duplicated(['quarter_id', 'activity_id', 'activity', 'gdp'], keep='first')
print(f'\nExact duplicate rows: {int(gdp["flag_exact_duplicate"].sum())}')
print(f'Quarters: {gdp["quarter"].replace("", np.nan).min()} to {gdp["quarter"].replace("", np.nan).max()}')

GDP cells that are not a number: 0
GDP cells empty: 0
GDP values zero or below: 0

Quarter IDs not in the form YYYYQ: 0
Quarter label agrees with the ID: 100.0%

Exact duplicate rows: 0
Quarters: 2010Q1 to 2026Q2


### Cell A3 — Activities, duplicates, missing quarters and big changes
- **Activities:** each GASTAT activity with its ID, number of quarters and first/last quarter.
- **Duplicates:** the same activity and quarter must have **one** value. Two different values usually mean two
  versions or price bases are mixed: the cell stops, because we cannot know which one is right.
- **Missing quarters:** quarters with no value for an activity, listed (nothing is filled in).
- **Big changes:** the 10 largest changes against the same quarter a year earlier. Real events (2020, oil price
  shocks) are expected and **kept**; only a value that is clearly a typing error would be flagged.

In [19]:
use = gdp[~gdp['flag_exact_duplicate'] & ~gdp['flag_bad_quarter']]
use = use.assign(qnum=use['qnum'].astype(int))

print('Activities:')
print(use.groupby(['activity_id', 'activity']).agg(
    quarters=('quarter', 'nunique'), first=('quarter', 'min'), last=('quarter', 'max'),
    empty_gdp=('flag_gdp_missing', 'sum')).to_string())

same_key = use.duplicated(['activity_id', 'quarter'], keep=False)
print(f'\nRows with two different values for the same activity and quarter: {int(same_key.sum())}')
if same_key.any():
    print(use[same_key].sort_values(['activity_id', 'qnum']).head(20).to_string(index=False))
assert not same_key.any(), 'Two values for one activity and quarter: choose one version / price basis in Cell A2.'

ALL_Q = pd.Index(range(int(use['qnum'].min()), int(use['qnum'].max()) + 1), name='qnum')
print('\nMissing quarters per activity:')
for activity, rows in use.groupby('activity'):
    gaps = sorted(set(ALL_Q) - set(rows.loc[~rows['flag_gdp_missing'], 'qnum']))
    print(f'   {activity[:60]:<60} {len(gaps):>3}', [qlabel(n) for n in gaps][:8] if gaps else '')

wide_all = use.pivot_table(index='qnum', columns='activity', values='gdp', aggfunc='first').reindex(ALL_Q)
yoy = (wide_all / wide_all.shift(4) - 1).replace([np.inf, -np.inf], np.nan)
changes = yoy.stack().rename('change vs a year earlier').reset_index()
changes['quarter'] = changes['qnum'].map(qlabel)
changes = changes.reindex(changes['change vs a year earlier'].abs().sort_values(ascending=False).index)
print('\n10 largest changes against the same quarter a year earlier:')
print(changes[['activity', 'quarter', 'change vs a year earlier']].head(10)
      .to_string(index=False, formatters={'change vs a year earlier': '{:+.1%}'.format}))

Activities:
                                                                 quarters   first    last  empty_gdp
activity_id activity                                                                                
1           Agriculture, Forestry & Fishing                            66  2010Q1  2026Q2          0
10          Government Services                                        66  2010Q1  2026Q2          0
11          Net Taxes on Products                                      66  2010Q1  2026Q2          0
12          Gross Domestic Product                                     66  2010Q1  2026Q2          0
2           Mining & Quarrying                                         66  2010Q1  2026Q2          0
3           Manufacturing                                              66  2010Q1  2026Q2          0
4           Electricity, Gas and Water                                 66  2010Q1  2026Q2          0
5           Construction                                               66  2010

**What we see →** _(fill in: number of activities, duplicates, gaps, are the big changes real events?)_

**Decision →** _(fill in)_

### Cell A4 — Mapping table: GASTAT activities → SSIC sections (the agreed table)
The GDP file has 12 activity groups; our businesses have one of the 21 SSIC sections (A–U). This is the table the
team agreed on (the same as `02_data_dictionary` Cell 9), by activity ID:

| ID | GASTAT activity | SSIC sections |
|---|---|---|
| 1 | Agriculture, Forestry & Fishing | A |
| 2 | Mining & Quarrying | B |
| 3 | Manufacturing | C |
| 4 | Electricity, Gas and Water | D, E |
| 5 | Construction | F |
| 6 | Wholesale & Retail Trade, Restaurants & hotels | G, I |
| 7 | Transport, Storage & Communication | H, J (telecom is "communication") |
| 8 | Finance, Insurance, Real Estate & Business Services | K, L, M, N |
| 9 | Community, Social & Personal Services | P, Q, R, S, T (private schools and clinics are here) |
| 10 | Government Services | O (the government itself) |
| 11, 12 | Net Taxes on Products, Gross Domestic Product | not an activity (the total is kept for each sector's share) |

A group that covers several sections gives all of them the same number. The cell stops if an ID is not in the table
or its name is not the expected one (the file changed): then check the table with the team.

In [20]:
# The agreed table (same as 02_data_dictionary Cell 9): activity ID -> (expected name, SSIC sections)
AGREED_GDP = {
    1: ('Agriculture, Forestry & Fishing', 'A'),
    2: ('Mining & Quarrying', 'B'),
    3: ('Manufacturing', 'C'),
    4: ('Electricity, Gas and Water', 'DE'),
    5: ('Construction', 'F'),
    6: ('Wholesale & Retail Trade, Restaurants & hotels', 'GI'),
    7: ('Transport, Storage & Communication', 'HJ'),
    8: ('Finance, Insurance, Real Estate & Business Services', 'KLMN'),
    9: ('Community, Social & Personal Services', 'PQRST'),
    10: ('Government Services', 'O'),
    11: ('Net Taxes on Products', 'not an activity'),
    12: ('Gross Domestic Product', 'not an activity'),
}

gdp_map = use[['activity_id', 'activity']].drop_duplicates().reset_index(drop=True)
gdp_map['id'] = pd.to_numeric(gdp_map['activity_id'], errors='coerce').astype('Int64')
gdp_map = gdp_map.sort_values('id').reset_index(drop=True)
unknown = gdp_map.loc[~gdp_map['id'].isin(list(AGREED_GDP)), 'activity'].tolist()
assert not unknown, f'Activity IDs not in the agreed table: {unknown}. Check the table with the team.'
expected = gdp_map['id'].map(lambda i: AGREED_GDP[int(i)][0])
renamed = gdp_map.loc[expected.str.lower() != gdp_map['activity'].str.lower(), ['id', 'activity']]
assert renamed.empty, f'Activity names differ from the agreed table:\n{renamed.to_string(index=False)}'
gdp_map['ssic_sections'] = gdp_map['id'].map(lambda i: AGREED_GDP[int(i)][1])
print(gdp_map[['activity_id', 'activity', 'ssic_sections']].to_string(index=False))

real = gdp_map[gdp_map['ssic_sections'] != 'not an activity']
section_activities = {}
for activity, sections in zip(real['activity'], real['ssic_sections']):
    for section in sections:
        section_activities.setdefault(section, []).append(activity)
not_covered = [s for s in SECTION_LETTERS if s not in section_activities]
print('\nSSIC sections with no GDP group (these businesses get an empty GDP feature + a flag):', not_covered or 'none')
print('Sections fed by more than one GASTAT group (values added up):',
      {s: a for s, a in section_activities.items() if len(a) > 1} or 'none')

activity_id                                            activity   ssic_sections
          1                     Agriculture, Forestry & Fishing               A
         10                                 Government Services               O
         11                               Net Taxes on Products not an activity
         12                              Gross Domestic Product not an activity
          2                                  Mining & Quarrying               B
          3                                       Manufacturing               C
          4                          Electricity, Gas and Water              DE
          5                                        Construction               F
          6      Wholesale & Retail Trade, Restaurants & hotels              GI
          7                  Transport, Storage & Communication              HJ
          8 Finance, Insurance, Real Estate & Business Services            KLMN
          9               Community, Soc

### Cell A5 — GDP features per SSIC section and start quarter
For a business that starts in quarter **Q**, only quarters **before Q** are used:

| Feature | Formula | Why |
|---|---|---|
| `gdp_growth_yoy` | GDP(Q−1) ÷ GDP(Q−5) − 1 | Growth against the same quarter a year earlier (removes the season) |
| `gdp_growth_4q` | sum of Q−4…Q−1 ÷ sum of Q−8…Q−5 − 1 | The same, over a full year: steadier |
| `gdp_share_4q` | sector's GDP in Q−4…Q−1 ÷ total GDP | How big the sector is in the economy |

**For founders on the website:** the table also covers the quarters after the last GDP quarter. They use the
latest quarter available, and `gdp_gap_quarters` says how many quarters old it is (0 for every training cohort).
A sector value is computed only when **all** its GASTAT groups have a value (never a partial sum).

In [21]:
acts_wide = wide_all[real['activity'].tolist()]
sector_wide = pd.DataFrame({s: acts_wide[a].sum(axis=1, min_count=len(a))
                            for s, a in sorted(section_activities.items())})
total_wide = acts_wide.sum(axis=1, min_count=acts_wide.shape[1])     # the sum of the real activities

last_q = int(sector_wide.dropna(how='all').index.max())
first_start = int(sector_wide.index.min()) + 8                       # needs 8 earlier quarters

rows = []
for start in range(first_start, last_q + 3):                         # up to 2 quarters after the last data
    used = min(start - 1, last_q)
    sum4 = sector_wide.loc[used - 3:used].sum(min_count=4)
    sum4_before = sector_wide.loc[used - 7:used - 4].sum(min_count=4)
    total4 = total_wide.loc[used - 3:used].sum(min_count=4)
    rows.append(pd.DataFrame({
        'start_quarter': qlabel(start),
        'ssic_section': sector_wide.columns,
        'gdp_quarter_used': qlabel(used),
        'gdp_gap_quarters': (start - 1) - used,
        'gdp_growth_yoy': (sector_wide.loc[used] / sector_wide.loc[used - 4] - 1).to_numpy(),
        'gdp_growth_4q': (sum4 / sum4_before - 1).to_numpy(),
        'gdp_share_4q': (sum4 / total4).to_numpy(),
    }))
gdp_features = pd.concat(rows, ignore_index=True).replace([np.inf, -np.inf], np.nan)
gdp_features['gdp_group'] = gdp_features['ssic_section'].map(lambda s: ' | '.join(section_activities[s]))

# Leakage check: every number comes from a quarter before the start quarter
assert (gdp_features['gdp_quarter_used'].map(qnum) < gdp_features['start_quarter'].map(qnum)).all()
print(f'Leakage check passed | {len(gdp_features):,} rows ({gdp_features["start_quarter"].nunique()} start quarters '
      f'x {gdp_features["ssic_section"].nunique()} sections) | last GDP quarter: {qlabel(last_q)}')
print('Empty values:', gdp_features[['gdp_growth_yoy', 'gdp_growth_4q', 'gdp_share_4q']].isna().sum().to_dict())

print('\ngdp_growth_4q for the labelled cohorts:')
print(gdp_features[gdp_features['start_quarter'].isin(COHORTS)]
      .pivot(index='ssic_section', columns='start_quarter', values='gdp_growth_4q').to_string(float_format='{:+.1%}'.format))

gdp_features.to_csv(DATA_DIR / 'gastat_gdp_features.csv', index=False, encoding='utf-8-sig')
print('\nSaved', DATA_DIR / 'gastat_gdp_features.csv')

Leakage check passed | 1,140 rows (60 start quarters x 19 sections) | last GDP quarter: 2026Q2
Empty values: {'gdp_growth_yoy': 0, 'gdp_growth_4q': 0, 'gdp_share_4q': 0}

gdp_growth_4q for the labelled cohorts:
start_quarter  2024Q4  2025Q1  2025Q2  2025Q3
ssic_section                                 
A               +6.7%   +5.1%   +4.1%   +3.3%
B              -10.3%   -6.2%   -3.2%   +0.9%
C               +1.4%   +5.0%   +5.4%   +5.5%
D               +5.6%   +6.6%   +6.1%   +6.3%
E               +5.6%   +6.6%   +6.1%   +6.3%
F               +4.5%   +5.0%   +5.4%   +5.2%
G               +7.4%   +6.8%   +6.8%   +6.4%
H               +4.4%   +4.3%   +4.5%   +4.3%
I               +7.4%   +6.8%   +6.8%   +6.4%
J               +4.4%   +4.3%   +4.5%   +4.3%
K               +7.0%   +7.1%   +6.6%   +6.0%
L               +7.0%   +7.1%   +6.6%   +6.0%
M               +7.0%   +7.1%   +6.6%   +6.0%
N               +7.0%   +7.1%   +6.6%   +6.0%
O               +3.0%   +3.3%   +3.5%   +2.6%
P      

**What we see →** _(fill in: do the growth numbers look reasonable for each sector?)_

## Part B — SME operating expenses and revenues

### Cell B1 — Read the file and check its columns
Expected columns (from `02_data_dictionary`): sector (as an SSIC section letter), establishment size (micro /
small / medium), flow (expenses / revenues), year (2019–2024) and value.

In [22]:
SME_HEADER = None          # write a row number here only if the guess below is wrong

raw = read_csv_raw(GASTAT_DIR / SME_FILE)
header = SME_HEADER if SME_HEADER is not None else find_header(raw)
print(f'Header row: {header}. Rows above it:')
print(raw.head(header).fillna('').to_string() if header else '   none')

sme_raw = with_header(raw, header)
print(f'\n{len(sme_raw):,} data rows | columns: {list(sme_raw.columns)}')

SME_COLUMNS = {'Economic Sectors ID': 'sector_id', 'Economic Sectors': 'sector',
               'Establishment Size ID': 'size_id', 'Establishment Size': 'size',
               'Flow ID': 'flow_id', 'Flow': 'flow', 'Year': 'year', 'Value': 'value'}
missing = set(SME_COLUMNS) - set(sme_raw.columns)
assert not missing, f'Columns not found: {missing}. Check the header row (SME_HEADER).'
extra = [c for c in sme_raw.columns if c not in SME_COLUMNS]
print('Other columns:', extra or 'none')
for column in extra:
    print(f'\n--- {column} ---')
    print(sme_raw[column].value_counts(dropna=False).head(10).to_string())
sme_raw.head(8)

Header row: 0. Rows above it:
   none

612 data rows | columns: ['Economic Sectors ID', 'Economic Sectors', 'Establishment Size ID', 'Establishment Size', 'Flow ID', 'Flow', 'Year', 'Value']
Other columns: none


,Economic Sectors ID,Economic Sectors,Establishment Size ID,Establishment Size,Flow ID,Flow,Year,Value
0,B,Mining and quarrying,1,Micro Establishments,1,Expenses,2019,33212.98645558239
1,B,Mining and quarrying,1,Micro Establishments,1,Expenses,2020,32227.817236180963
2,B,Mining and quarrying,1,Micro Establishments,1,Expenses,2021,35724.516635888525
3,B,Mining and quarrying,1,Micro Establishments,1,Expenses,2022,36687.727205002564
4,B,Mining and quarrying,1,Micro Establishments,1,Expenses,2023,458678.15729066846
5,B,Mining and quarrying,1,Micro Establishments,1,Expenses,2024,472325.736297092
6,B,Mining and quarrying,1,Micro Establishments,2,Revenues,2019,108029.8812
7,B,Mining and quarrying,1,Micro Establishments,2,Revenues,2020,103764.77779412782


### Cell B2 — Data types, codes, duplicates and missing combinations
1. `Value` and `Year` text → numbers (cells that fail are shown).
2. Every ID must match one name (e.g. size `1` = micro). The sector ID must be an SSIC section letter.
3. **Duplicates:** exact copies are flagged; two different values for the same sector, size, flow and year stop the cell.
4. **Missing combinations:** every sector should have every size, flow and year. Missing ones are listed, not filled.

In [23]:
sme = sme_raw[list(SME_COLUMNS)].rename(columns=SME_COLUMNS).copy()
for column in ['sector_id', 'sector', 'size_id', 'size', 'flow_id', 'flow']:
    sme[column] = clean_text(sme[column])
sme['sector_id'] = sme['sector_id'].str.upper()

sme['year'], bad_year = to_number(sme['year'])
sme['value'], bad_value = to_number(sme['value'])
print(f'Year not a number: {int(bad_year.sum())} | Value not a number: {int(bad_value.sum())}')
if bad_value.any():
    print(sme_raw.loc[bad_value[bad_value].index, 'Value'].value_counts().head(10).to_string())
sme['flag_value_missing'] = sme['value'].isna()
print(f'Value empty: {int(sme["flag_value_missing"].sum())} | zero or below: {int((sme["value"] <= 0).sum())}')

# 2. IDs and names
for id_column, name_column in [('sector_id', 'sector'), ('size_id', 'size'), ('flow_id', 'flow')]:
    pairs = sme.groupby([id_column, name_column]).size().rename('rows').reset_index()
    print(f'\n{id_column} -> {name_column}:')
    print(pairs.to_string(index=False))
    names_per_id = pairs.groupby(id_column)[name_column].nunique()
    if (names_per_id > 1).any():
        print('   !! IDs with more than one name:', names_per_id[names_per_id > 1].index.tolist())
bad_sector = ~sme['sector_id'].isin(SECTION_LETTERS)
print(f'\nSector IDs that are not an SSIC section letter: {sorted(sme.loc[bad_sector, "sector_id"].unique())}')
print('SSIC sections not in the file:', [s for s in SECTION_LETTERS if s not in set(sme['sector_id'])])

# 3. Duplicates
KEY = ['sector_id', 'size_id', 'flow_id', 'year']
sme['flag_exact_duplicate'] = sme.duplicated(KEY + ['value'], keep='first')
use_sme = sme[~sme['flag_exact_duplicate']]
same_key = use_sme.duplicated(KEY, keep=False)
print(f'\nExact duplicate rows: {int(sme["flag_exact_duplicate"].sum())} | '
      f'two different values for one key: {int(same_key.sum())}')
assert not same_key.any(), 'Two values for the same sector, size, flow and year: check the file.'

# 4. Missing combinations
grid = pd.MultiIndex.from_product([sorted(use_sme[c].dropna().unique()) for c in KEY], names=KEY)
present = pd.MultiIndex.from_frame(use_sme.loc[use_sme['value'].notna(), KEY])
absent = grid.difference(present)
print(f'Combinations expected: {len(grid):,} | with a value: {len(present):,} | missing: {len(absent):,}')
if len(absent):
    print(absent.to_frame(index=False).head(20).to_string(index=False))

Year not a number: 0 | Value not a number: 0
Value empty: 0 | zero or below: 0

sector_id -> sector:
sector_id                                                               sector  rows
        B                                                 Mining and quarrying    36
        C                                                        Manufacturing    36
        D                  Electricity, gas, steam and air conditioning supply    36
        E   Water supply sewerage, waste management and remediation activities    36
        F                                                         Construction    36
        G Wholesale and retail trade; repair of motor vehicles and motorcycles    36
        H                                           Transportation and storage    36
        I                            Accommodation and food service activities    36
        J                                        Information and communication    36
        K                                   Finan

**What we see →** _(fill in)_

**Decision →** _(fill in)_

### Cell B3 — Big jumps between years
`02_data_dictionary` noticed that 2023 values jump for some sectors. This cell lists the largest changes from one
year to the next and counts the jumps above ±50% per year.

**Decision rule:** a jump is **kept** (a real change in the sector, or a change in how GASTAT counts). It is not
capped or smoothed: our features use ratios and growth, and capping would invent numbers. If one year jumps for
almost every sector at once, that points to a change in method: note it, because growth into that year is then
not comparable.

In [24]:
by_year = (use_sme.pivot_table(index=['sector_id', 'size', 'flow'], columns='year', values='value', aggfunc='first')
           .sort_index(axis=1))
change = (by_year / by_year.shift(axis=1) - 1).replace([np.inf, -np.inf], np.nan)

print('Changes above ±50% per year (all sectors, sizes and flows):')
print((change.abs() > 0.5).sum().rename('jumps').to_string())

top = change.stack().rename('change').reset_index()
top = top.reindex(top['change'].abs().sort_values(ascending=False).index).head(15)
print('\n15 largest changes from the year before:')
print(top.to_string(index=False, formatters={'change': '{:+.0%}'.format}))

Changes above ±50% per year (all sectors, sizes and flows):
year
2019.0     0
2020.0     3
2021.0    14
2022.0     0
2023.0    21
2024.0     0

15 largest changes from the year before:
sector_id                  size     flow   year change
        D  Micro Establishments Expenses 2023.0 +6798%
        D  Micro Establishments Revenues 2023.0 +6155%
        D  Small Establishments Expenses 2023.0 +1981%
        B  Micro Establishments Expenses 2023.0 +1150%
        D  Small Establishments Revenues 2023.0  +912%
        B  Micro Establishments Revenues 2023.0  +894%
        L  Small Establishments Expenses 2023.0  +316%
        L  Micro Establishments Expenses 2023.0  +227%
        L  Micro Establishments Revenues 2023.0  +226%
        L  Small Establishments Revenues 2023.0  +212%
        L Medium Establishments Expenses 2021.0  +201%
        R Medium Establishments Revenues 2021.0  +132%
        R Medium Establishments Expenses 2021.0  +129%
        J Medium Establishments Expenses 2023

**What we see →** _(fill in: is the 2023 jump in one sector or in almost all of them?)_

**Decision →** _(fill in)_

### Cell B4 — SME features per SSIC section and start year
The MC files have no size column, and almost every new business is **micro**, so the micro rows are used.
For a business that starts in year **Y**, only year **Y − 1** is used:

| Feature | Formula | Why |
|---|---|---|
| `sme_rev_exp_ratio` | revenues ÷ expenses of micro businesses in the sector, year Y−1 | Above 1 = the sector's small businesses earn more than they spend |
| `sme_revenue_growth` | revenues(Y−1) ÷ revenues(Y−2) − 1 | Is the sector's small-business market growing? |

Expenses or revenues of zero or below give an empty value (no division by zero). Founders in years after the last
data use the latest year; `sme_gap_years` says how old it is (0 for the training cohorts, which start in 2024–2025).

In [25]:
size_names = use_sme.drop_duplicates('size_id').set_index('size_id')['size']
flow_names = use_sme.drop_duplicates('flow_id').set_index('flow_id')['flow']
MICRO    = [i for i, n in size_names.items() if 'micro' in n.lower()]
EXPENSES = [i for i, n in flow_names.items() if 'expens' in n.lower()]
REVENUES = [i for i, n in flow_names.items() if 'revenu' in n.lower()]
assert len(MICRO) == 1 and len(EXPENSES) == 1 and len(REVENUES) == 1, \
    f'Could not find micro / expenses / revenues by name: {size_names.to_dict()} {flow_names.to_dict()}'

micro = use_sme[(use_sme['size_id'] == MICRO[0]) & use_sme['sector_id'].isin(SECTION_LETTERS) & use_sme['year'].notna()]
micro = micro.assign(year=micro['year'].astype(int))
years = range(int(micro['year'].min()), int(micro['year'].max()) + 1)

def flow_table(flow_id):
    return micro[micro['flow_id'] == flow_id].pivot_table(index='year', columns='sector_id', values='value', aggfunc='first')

revenues, expenses = flow_table(REVENUES[0]), flow_table(EXPENSES[0])
sectors = revenues.columns.union(expenses.columns)
revenues = revenues.reindex(index=years, columns=sectors)
expenses = expenses.reindex(index=years, columns=sectors)
last_year = int(revenues.dropna(how='all').index.max())

rows = []
for start_year in range(years[0] + 2, last_year + 3):
    used = min(start_year - 1, last_year)
    positive_expenses = expenses.loc[used].where(expenses.loc[used] > 0)
    positive_revenues_before = revenues.loc[used - 1].where(revenues.loc[used - 1] > 0)
    rows.append(pd.DataFrame({
        'start_year': start_year,
        'ssic_section': sectors,
        'sme_year_used': used,
        'sme_gap_years': (start_year - 1) - used,
        'sme_rev_exp_ratio': (revenues.loc[used] / positive_expenses).to_numpy(),
        'sme_revenue_growth': (revenues.loc[used] / positive_revenues_before - 1).to_numpy(),
    }))
sme_features = pd.concat(rows, ignore_index=True).replace([np.inf, -np.inf], np.nan)

# Leakage check: every number comes from a year before the start year
assert (sme_features['sme_year_used'] < sme_features['start_year']).all()
print(f'Leakage check passed | {len(sme_features):,} rows | last SME year: {last_year}')
print('Empty values:', sme_features[['sme_rev_exp_ratio', 'sme_revenue_growth']].isna().sum().to_dict())
print('Sections with no SME data (empty feature + flag):', [s for s in SECTION_LETTERS if s not in sectors])

print('\nRevenue / expense ratio used by each training cohort year:')
print(sme_features[sme_features['start_year'].isin([2024, 2025])]
      .pivot(index='ssic_section', columns='start_year', values='sme_rev_exp_ratio').to_string(float_format='{:.2f}'.format))

sme_features.to_csv(DATA_DIR / 'gastat_sme_features.csv', index=False, encoding='utf-8-sig')
print('\nSaved', DATA_DIR / 'gastat_sme_features.csv')

Leakage check passed | 102 rows | last SME year: 2024
Empty values: {'sme_rev_exp_ratio': 0, 'sme_revenue_growth': 0}
Sections with no SME data (empty feature + flag): ['A', 'O', 'T', 'U']

Revenue / expense ratio used by each training cohort year:
start_year    2024  2025
ssic_section            
B             2.52  2.55
C             3.02  3.04
D             2.65  2.69
E             2.11  2.14
F             2.45  2.48
G             1.94  1.98
H             2.50  2.53
I             2.21  2.14
J             2.50  2.40
K             2.35  2.36
L             2.84  2.81
M             2.98  2.96
N             3.07  3.03
P             2.75  2.66
Q             2.72  2.80
R             2.65  2.81
S             2.22  2.26

Saved /content/darb-data/gastat_sme_features.csv


## Part C — SSIC: activities that need a license outside the Ministry of Commerce

### Cell C1 — Read the SSIC file
The sheet starts with a title; the real header is the row that starts with "رمز الباب" (same method as
`01_preprocessing` Cell 16). Codes are read **as text** and padded (division `1` → `01`), so no leading zero is lost.
If the file has merged cells, the section / division / group / class are carried down to every activity row.

In [26]:
raw = pd.read_excel(GASTAT_DIR / SSIC_FILE, engine='calamine', header=None, dtype=str)
header_row = raw.index[raw[0].fillna('').str.strip() == 'رمز الباب'][0]
ssic = raw.iloc[header_row + 1:].copy()

SSIC_NAMES = ['section_code', 'section_name', 'division_code', 'division_name', 'group_code', 'group_name',
              'class_code', 'class_name', 'activity_code', 'activity_name', 'activity_description',
              'supervising_authority', 'licensing_authority']
assert ssic.shape[1] == len(SSIC_NAMES), f'The SSIC sheet has {ssic.shape[1]} columns, expected {len(SSIC_NAMES)}.'
ssic.columns = SSIC_NAMES
ssic = ssic.apply(lambda column: column.str.strip())

LEVELS = ['section_code', 'section_name', 'division_code', 'division_name', 'group_code', 'group_name', 'class_code', 'class_name']
empty_before = ssic[LEVELS].isna().sum()
ssic[LEVELS] = ssic[LEVELS].ffill()                  # merged cells -> value on every row
ssic = ssic.dropna(subset=['activity_code']).reset_index(drop=True)
print('Empty level cells filled from the row above (merged cells):', empty_before[empty_before > 0].to_dict() or 'none')

for column, width in [('division_code', 2), ('group_code', 3), ('class_code', 4), ('activity_code', 6)]:
    ssic[column] = clean_text(ssic[column]).str.zfill(width)

# Arabic section letters -> A ... U with a FIXED table (the abjad order SSIC uses), not "the n-th letter found":
# the file has only 20 of the 21 sections, so counting them would shift every letter after the missing one.
ARABIC_TO_LATIN = {'ا': 'A', 'ب': 'B', 'ج': 'C', 'د': 'D', 'ه': 'E', 'و': 'F', 'ز': 'G', 'ح': 'H', 'ط': 'I', 'ي': 'J',
                   'ك': 'K', 'ل': 'L', 'م': 'M', 'ن': 'N', 'س': 'O', 'ع': 'P', 'ف': 'Q', 'ص': 'R', 'ق': 'S', 'ر': 'T', 'ش': 'U'}
letter = (ssic['section_code'].str.replace(r'[ـ\s]', '', regex=True)          # 'هـ' -> 'ه'
          .str.replace('[أإآ]', 'ا', regex=True).str.replace('ى', 'ي'))
ssic['ssic_section'] = letter.map(ARABIC_TO_LATIN)
unknown = sorted(ssic.loc[ssic['ssic_section'].isna(), 'section_code'].unique())
assert not unknown, f'Section codes not in ARABIC_TO_LATIN: {unknown}'
print(ssic.drop_duplicates('ssic_section')[['ssic_section', 'section_code', 'section_name']]
      .sort_values('ssic_section').to_string(index=False))
print('SSIC sections not in this file:', [s for s in SECTION_LETTERS if s not in set(ssic['ssic_section'])])

# Check on divisions whose section is known (ISIC 4): each must get the right letter
check = ssic.drop_duplicates('division_code').set_index('division_code')['ssic_section']
KNOWN = {'01': 'A', '10': 'C', '41': 'F', '47': 'G', '56': 'I', '62': 'J', '84': 'O', '85': 'P', '86': 'Q', '99': 'U'}
wrong = {d: check[d] for d, l in KNOWN.items() if d in check and check[d] != l}
assert not wrong, f'Divisions in the wrong section: {wrong}. Check ARABIC_TO_LATIN.'

print(f'\n{len(ssic):,} activities | {ssic["ssic_section"].nunique()} sections | {ssic["division_code"].nunique()} divisions')
print(f'Activity codes that repeat: {int(ssic["activity_code"].duplicated().sum())} | '
      f'not 6 digits: {int((~ssic["activity_code"].str.fullmatch(r"[0-9]{6}")).sum())}')
print(f'Activities with no licensing authority written: {int(ssic["licensing_authority"].isna().sum())}')

Empty level cells filled from the row above (merged cells): {'section_name': 1, 'division_code': 1, 'division_name': 1, 'group_code': 1, 'group_name': 1, 'class_code': 1, 'class_name': 1}
ssic_section section_code                                                         section_name
           A            أ                                        الزراعة والحراجة وصيد الأسماك
           B            ب                                             التعدين واستغلال المحاجر
           C            ج                                                    الصناعة التحويلية
           D            د                        إمدادات الكهرباء والغاز والبخار وتكييف الهواء
           E           هـ          إمدادات المياه وأنشطة الصرف الصحي وإدارة النفايات ومعالجتها
           F            و                                                              التشييد
           G            ز تجارة الجملة والتجزئة وإصلاح المركبات ذات المحركات والدراجات النارية
           H            ح                           

### Cell C2 — Which authority licenses each activity
The licensing authority is free text. An activity counts as **licensed outside the Ministry of Commerce** when its
text still names something after "وزارة التجارة" is removed (so "وزارة التجارة، وزارة الصحة" counts as outside).
An empty cell is **not** counted as either: it gets its own share (`licensing_not_stated_share`).

The 20 most common texts and some examples of each decision are printed: **check them by eye**.

In [27]:
licensing = ssic['licensing_authority'].fillna('').str.replace(r'\s+', ' ', regex=True).str.strip()
print('20 most common licensing authorities:')
print(licensing.value_counts().head(20).to_string())

MC_NAME = r'وزار[ةه] التجار[ةه]( والاستثمار)?'
rest = licensing.str.replace(MC_NAME, '', regex=True).str.replace(r'[،,;/\-–()]', ' ', regex=True)
rest = rest.str.replace(r'^\s*و\s+|\s+و\s*$', ' ', regex=True).str.strip()
ssic['licensing_not_stated'] = licensing == ''
ssic['licensed_outside_mc'] = ~ssic['licensing_not_stated'] & (rest.str.len() > 2)

print('\nExamples counted as OUTSIDE the Ministry of Commerce:')
print(licensing[ssic['licensed_outside_mc']].drop_duplicates().head(8).to_string(index=False))
print('\nExamples counted as Ministry of Commerce only:')
print(licensing[~ssic['licensed_outside_mc'] & ~ssic['licensing_not_stated']].drop_duplicates().head(8).to_string(index=False))
print(f'\nActivities licensed outside MC: {ssic["licensed_outside_mc"].mean():.1%} | '
      f'not stated: {ssic["licensing_not_stated"].mean():.1%}')

20 most common licensing authorities:
licensing_authority
وزارة الصناعة والثروة المعدنية               880
وزارة البلديات والإسكان                      490
الهيئة العامة للغذاء والدواء                 258
وزارة التجارة                                169
وزارة البيئة والمياه والزراعة                131
الهيئة العامة للصناعات العسكرية               78
الهيئة العامة للنقل                           66
هيئة الاتصالات والفضاء والتقنية               59
الهيئة العامة للطيران المدني                  48
وزارة التعليم                                 44
وزارة الصحة                                   43
وزارة الطاقة                                  37
وزارة الإعلام                                 33
المركز الوطني لإدارة النفايات                 33
الهيئة العليا للأمن الصناعي                   32
الهيئة العامة لتنظيم الاعلام                  30
البنك المركزي السعودي                         29
وزارة الثقافة                                 28
وزارة الموارد البشرية والتنمية الإجتماعية     26
المركز الوط

### Cell C3 — Licensing features per division and per section
One row per SSIC division: number of activities and the share that need a license outside the Ministry of
Commerce. A section-level table is saved too, for businesses whose division is not known (`ssic_flag =
division_unclear`). These are facts about the activity, not about time, so there is no leakage.

In [28]:
def licensing_table(keys):
    return (ssic.groupby(keys).agg(n_activities=('activity_code', 'size'),
                                   licensed_outside_mc_share=('licensed_outside_mc', 'mean'),
                                   licensing_not_stated_share=('licensing_not_stated', 'mean'))
            .reset_index())

division_features = licensing_table(['ssic_section', 'division_code', 'division_name']).rename(columns={'division_code': 'ssic_division'})
section_features = licensing_table(['ssic_section', 'section_name'])

print('Divisions where most activities need a license outside MC:')
print(division_features.sort_values('licensed_outside_mc_share', ascending=False).head(15)
      .to_string(index=False, formatters={'licensed_outside_mc_share': '{:.0%}'.format, 'licensing_not_stated_share': '{:.0%}'.format}))
print('\nPer section:')
print(section_features.to_string(index=False, formatters={'licensed_outside_mc_share': '{:.0%}'.format, 'licensing_not_stated_share': '{:.0%}'.format}))

division_features.to_csv(DATA_DIR / 'gastat_ssic_division_features.csv', index=False, encoding='utf-8-sig')
section_features.to_csv(DATA_DIR / 'gastat_ssic_section_features.csv', index=False, encoding='utf-8-sig')
print('\nSaved', DATA_DIR / 'gastat_ssic_division_features.csv', 'and', DATA_DIR / 'gastat_ssic_section_features.csv')

Divisions where most activities need a license outside MC:
ssic_section ssic_division                                                                       division_name  n_activities licensed_outside_mc_share licensing_not_stated_share
           A            01                      أنشطة زراعة المحاصيل والإنتاج الحيواني والصيد والخدمات المتصلة            59                      100%                         0%
           A            02                                                                الحراجة وقطع الأخشاب             6                      100%                         0%
           A            03                                                         صيد الأسماك وتربية المائيات            14                      100%                         0%
           B            05                                                               تعدين الفحم والليغنيت             3                      100%                         0%
           B            06                         

## Part D — Join test with the businesses

### Cell D1 — Attach the GASTAT features to every business
Needs two files from earlier notebooks (on Colab: run them first on the **same server**, or upload the files to
`/content/darb-data`):
- `ssic_by_business.csv` (`01_preprocessing` Cell 19): one row per business with its cohort and label.
- `ssic_keywords_by_business.csv` (`03_official_activity` Cell 14): the business's **sector**, from the start name
  with the method "keywords, then rules" (`keywords_then_rules_section` / `keywords_then_rules_division`).

How each feature is attached:
- **GDP:** by `cohort` (= start quarter) and the section.
- **SME:** by the start year (from `cohort`) and the section.
- **Licensing:** by section + division; when the division is unknown, the section's share is used and
  `licensing_from_section` = True.

A business with no sector (`"unclassified"` in the model) gets **no** GASTAT feature (empty, never guessed). Every
feature has a `has_…` flag. The **coverage** is printed for the labelled businesses, and split by the label: it must
be about the same for businesses that survived and those that disappeared, otherwise the missing values themselves
would tell the model the answer.

Saves `gastat_features_by_business.csv` (one row per business), never in Git.

In [29]:
BUSINESSES = DATA_DIR / 'ssic_by_business.csv'          # 01_preprocessing Cell 19: cohort and label
SECTORS = DATA_DIR / 'ssic_keywords_by_business.csv'    # 03_official_activity Cell 14: the sector
missing_files = [p.name for p in (BUSINESSES, SECTORS) if not p.exists()]
if missing_files:
    print('Not found:', missing_files, '-> run 01_preprocessing (Cells 1-19) and 03_official_activity first, '
          f'on the same server (or upload the files to {DATA_DIR}).')
else:
    business = pd.read_csv(BUSINESSES, dtype=str, encoding='utf-8-sig',
                           usecols=['entity_id', 'cohort', 'cohort_quality', 'survived_1y']).fillna('')
    sectors = (pd.read_csv(SECTORS, dtype=str, encoding='utf-8-sig',
                           usecols=['entity_id', 'keywords_then_rules_section', 'keywords_then_rules_division'])
               .rename(columns={'keywords_then_rules_section': 'ssic_section',
                                'keywords_then_rules_division': 'ssic_division'}))
    business = business.merge(sectors, on='entity_id', how='left')
    business[['ssic_section', 'ssic_division']] = business[['ssic_section', 'ssic_division']].fillna('')
    business['ssic_division'] = business['ssic_division'].where(business['ssic_division'] == '',
                                                                business['ssic_division'].str.zfill(2))
    business['start_year'] = pd.to_numeric(business['cohort'].str[:4], errors='coerce').astype('Int64')

    out = business.merge(gdp_features.drop(columns=['gdp_group']).rename(columns={'start_quarter': 'cohort'}),
                         on=['cohort', 'ssic_section'], how='left')
    out = out.merge(sme_features.assign(start_year=sme_features['start_year'].astype('Int64')),
                    on=['start_year', 'ssic_section'], how='left')
    out = out.merge(division_features[['ssic_section', 'ssic_division', 'licensed_outside_mc_share']],
                    on=['ssic_section', 'ssic_division'], how='left')
    section_share = section_features.set_index('ssic_section')['licensed_outside_mc_share']
    out['licensing_from_section'] = out['licensed_outside_mc_share'].isna() & (out['ssic_section'] != '')
    out.loc[out['licensing_from_section'], 'licensed_outside_mc_share'] = (
        out.loc[out['licensing_from_section'], 'ssic_section'].map(section_share))

    out['has_sector'] = out['ssic_section'] != ''
    out['has_gdp'] = out['gdp_growth_4q'].notna()
    out['has_sme'] = out['sme_rev_exp_ratio'].notna()
    out['has_licensing'] = out['licensed_outside_mc_share'].notna()
    assert out['entity_id'].is_unique, 'A business got more than one row: a feature table has a repeated key.'

    labelled = out[(out['cohort_quality'] != 'cannot_judge') & out['survived_1y'].isin(['0', '1'])]
    FLAGS = ['has_sector', 'has_gdp', 'has_sme', 'has_licensing']
    print(f'Businesses: {len(out):,} | labelled (0 or 1): {len(labelled):,}')
    print('\nCoverage of the labelled businesses:')
    print(labelled[FLAGS].mean().map('{:.1%}'.format).to_string())
    print('\nCoverage of the labelled businesses that HAVE a sector:')
    print(labelled.loc[labelled['has_sector'], FLAGS[1:]].mean().map('{:.1%}'.format).to_string())
    print('\nCoverage by label (should be about the same for 0 and 1):')
    print(labelled.groupby('survived_1y')[FLAGS].mean().to_string(float_format='{:.1%}'.format))

    out.to_csv(DATA_DIR / 'gastat_features_by_business.csv', index=False, encoding='utf-8-sig')
    print('\nSaved', DATA_DIR / 'gastat_features_by_business.csv')

ssic_by_business.csv not found: run 01_preprocessing Cell 19 first (on Colab: upload it to /content/darb-data).


### Notes — results and open points (fill in after running)
- **GDP:** price basis used (current / constant): __. Mapping: the agreed table (Cell A4, same as `02_data_dictionary`
  Cell 9): all 12 IDs and names match. Sections with no GDP group: U only (no business is in U).
- **SME:** the 2023 jump is __ (one sector / almost all sectors → change of method?). Sections with no data: A, O, T, U (1% of businesses with a sector).
- **Licensing:** share of activities licensed outside MC: __. Is the "outside MC" rule right on the printed examples? __
- **Sector:** from `03_official_activity` ("keywords, then rules", start name). Businesses with no sector get no
  GASTAT feature (the model treats them as `"unclassified"`).
- **Coverage (labelled businesses, 493,065):** sector 45.5%, GDP 45.5%, SME 45.0%, licensing 45.5%. Of those with a
  sector: GDP 100%, SME 99%, licensing 100%. Survived vs disappeared: sector 45.5% vs 44.9%, GDP 45.5% vs 44.9%,
  SME 45.1% vs 44.4% → about the same (no leakage through missing values).
- **Units:** not stated in the files. All features here are growth rates, ratios and shares, so they do **not**
  depend on the unit. The unit is only needed to describe the data in the report.
- **Revisions:** GASTAT revises recent quarters later. The file is the version of 21 Oct 2026; the model is
  trained on that version.
- **Saved files** (data folder, never in Git): `gastat_gdp_features.csv`, `gastat_sme_features.csv`,
  `gastat_ssic_division_features.csv`, `gastat_ssic_section_features.csv`, `gastat_features_by_business.csv`.